In [2]:
from google.colab import drive
import sys

drive.mount("/content/drive")
sys.path.append("/content/drive/MyDrive/colab_env/lib/python3.11/site-packages")

# !source /content/drive/MyDrive/colab_env/bin/activate; pip install Pypdf

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import math
import os
import pprint

import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
print('TF: {}'.format(tf.__version__))

# import tensorflow_data_validation as tfdv
# print('TFDV version:', tfdv.version.__version__)

import apache_beam as beam
print('Beam: {}'.format(beam.__version__))

import tensorflow_transform as tft
import tensorflow_transform.beam as tft_beam
from tensorflow_transform.keras_lib import tf_keras
print('Transform: {}'.format(tft.__version__))

from tfx_bsl.public import tfxio
from tfx_bsl.coders.example_coder import RecordBatchToExamplesEncoder

tf.compat.v1.disable_eager_execution()

TF: 2.18.0
Beam: 2.65.0
Transform: 1.16.0


In [20]:
df = pd.read_csv("/content/drive/MyDrive/laptop_data_2.csv")
df.columns

Index(['Company', 'TypeName', 'Inches', 'ScreenResolution', 'Cpu', 'Ram',
       'Memory', 'Gpu', 'OpSys', 'Weight', 'Price'],
      dtype='object')

In [21]:
for column in df.columns:
    print(f"========== VALUE COUNTS: [{column}] =================")
    print(df[column].value_counts(dropna=False))
    print(f"NULL values: {df[column].isnull().sum()}")

========== VALUE COUNTS: [Company] =================
Company
MSI       1429821
Apple     1428790
Lenovo    1428689
Acer      1428659
HP        1428487
Asus      1427974
Dell      1427580
Name: count, dtype: int64
NULL values: 0
========== VALUE COUNTS: [TypeName] =================
TypeName
Ultrabook             2001264
Gaming                2000488
Netbook               2000434
Notebook              1999142
2 in 1 Convertible    1998672
Name: count, dtype: int64
NULL values: 0
========== VALUE COUNTS: [Inches] =================
Inches
12.4    159841
15.1    159389
13.1    159379
16.7    159363
16.9    159354
         ...  
14.9    158039
11.8    157872
15.3    157675
17.3     79096
11.0     78806
Name: count, Length: 64, dtype: int64
NULL values: 0
========== VALUE COUNTS: [ScreenResolution] =================
ScreenResolution
3840x2160    2502743
1366x768     2499912
1920x1080    2498838
3200x1800    2498507
Name: count, dtype: int64
NULL values: 0
========== VALUE COUNTS: [Cpu] ======

In [22]:
df[df["Gpu"].isnull()]

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
455,Acer,Ultrabook,14.2,1366x768,Intel Celeron,8GB,2TB SSD,NaN,Windows 11,1.2kg,1781.23
646,HP,Ultrabook,17.2,3840x2160,Intel Celeron,4GB,256GB SSD,NaN,Windows 10,1.2kg,1280.0
671,Dell,Gaming,17.3,3840x2160,Intel Core i5,32GB,1TB HDD,NaN,Windows 11,2.0kg,2723.2
712,Asus,Gaming,11.4,3840x2160,Apple M1,8GB,128GB SSD,NaN,Windows 11,2.5kg,1757.14
1114,Acer,Gaming,11.1,3840x2160,Apple M1,32GB,256GB SSD,NaN,Windows 11,1.2kg,1361.26
...,...,...,...,...,...,...,...,...,...,...,...
9998955,Asus,2 in 1 Convertible,11.3,3200x1800,Intel Core i5,32GB,512GB SSD,NaN,Windows 11,1.5kg,2204.75
9998984,Dell,2 in 1 Convertible,12.4,1920x1080,Intel Core i5,32GB,2TB SSD,NaN,macOS,2.0kg,2937.43
9999424,Lenovo,Gaming,11.6,1920x1080,Apple M1,4GB,256GB SSD,NaN,Windows 11,2.5kg,2066.64
9999440,MSI,2 in 1 Convertible,12.5,1920x1080,Intel Celeron,8GB,1TB HDD,NaN,Windows 11,2.0kg,469.34


In [23]:
df["Gpu"].value_counts(dropna=False)

,count
Gpu,
AMD Radeon,2476449
NVIDIA GTX 1650,2475776
Intel UHD,2474135
NVIDIA RTX 3060,2473640
###invalid###,50084
NaN,49916


In [8]:
# df = pd.read_csv("sample.csv")  # or sample.csv if used during analysis
# mean = df["Price"].mean()
# std = df["Price"].std()

# print("Mean:", mean)
# print("Std:", std)

In [25]:
from sklearn.model_selection import train_test_split
import numpy as np

NUM_CHUNKS = 10

# First split: train and temp (eval + test)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# Second split: eval and test from temp
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Sample 10% of train data for Beam
sample_df = train_df.sample(frac=0.3, random_state=42)

print(f"Sample: {len(sample_df)} rows (10% of train)")
print(f"Eval: {len(eval_df)} rows")
print(f"Test: {len(test_df)} rows")

# Split and write into 10 chunks
chunks = np.array_split(train_df, NUM_CHUNKS)
for i, chunk in enumerate(chunks):
    print(f"Train_{i}: {len(chunk)} rows")
    chunk.to_csv(f"train_{i}.csv", index=False)

eval_df.to_csv('eval.csv', index=False)
test_df.to_csv('test.csv', index=False)
sample_df.to_csv("sample.csv", index=False)


Sample: 2100000 rows (10% of train)
Eval: 1500000 rows
Test: 1500000 rows
Train_0: 700000 rows
Train_1: 700000 rows
Train_2: 700000 rows
Train_3: 700000 rows
Train_4: 700000 rows
Train_5: 700000 rows
Train_6: 700000 rows
Train_7: 700000 rows
Train_8: 700000 rows
Train_9: 700000 rows


In [ ]:
# 📊 Step 5: Generate statistics
train_stats = tfdv.generate_statistics_from_csv(data_location='train.csv')
eval_stats = tfdv.generate_statistics_from_csv(data_location='eval.csv')
test_stats = tfdv.generate_statistics_from_csv(data_location='test.csv')

# 🧠 Step 6: Infer schema from training stats
schema = tfdv.infer_schema(statistics=train_stats)
tfdv.display_schema(schema)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


,Type,Presence,Valency,Domain
Feature name,,,,
'Company',STRING,required,,'Company'
'TypeName',STRING,required,,'TypeName'
'Inches',FLOAT,required,,-
'ScreenResolution',STRING,required,,'ScreenResolution'
'Cpu',STRING,required,,'Cpu'
'Ram',STRING,optional,single,'Ram'
'Memory',STRING,required,,'Memory'
'Gpu',STRING,optional,single,'Gpu'
'OpSys',STRING,required,,'OpSys'


,Values
Domain,
'Company',"'Acer', 'Apple', 'Asus', 'Dell', 'HP', 'Lenovo', 'MSI'"
'TypeName',"'2 in 1 Convertible', 'Gaming', 'Netbook', 'Notebook', 'Ultrabook'"
'ScreenResolution',"'1366x768', '1920x1080', '3200x1800', '3840x2160'"
'Cpu',"'AMD Ryzen 5', 'Apple M1', 'Intel Celeron', 'Intel Core i5', 'Intel Core i7'"
'Ram',"'1234TB', '16GB', '32GB', '4GB', '8GB'"
'Memory',"'128GB SSD', '1TB HDD', '256GB SSD', '2TB SSD', '512GB SSD'"
'Gpu',"'###invalid###', 'AMD Radeon', 'Intel UHD', 'NVIDIA GTX 1650', 'NVIDIA RTX 3060'"
'OpSys',"'Linux', 'Windows 10', 'Windows 11', 'macOS'"
'Weight',"'1.2kg', '1.5kg', '2.0kg', '2.5kg'"


In [ ]:
# 🚨 Step 7: Validate eval/test stats against schema
eval_anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)
test_anomalies = tfdv.validate_statistics(statistics=test_stats, schema=schema)

# 👁️ Step 8: Display schema validation anomalies
print("🔍 Anomalies in eval.csv:")
tfdv.display_anomalies(eval_anomalies)

print("🔍 Anomalies in test.csv:")
tfdv.display_anomalies(test_anomalies)

🔍 Anomalies in eval.csv:


🔍 Anomalies in test.csv:


In [ ]:
# 🚨 Validate eval and test using schema + compare to train statistics (for drift)
eval_anomalies = tfdv.validate_statistics(
    statistics=eval_stats,
    schema=schema,
    previous_statistics=train_stats  # for drift detection
)

test_anomalies = tfdv.validate_statistics(
    statistics=test_stats,
    schema=schema,
    previous_statistics=train_stats
)

# 🔍 Show schema + data drift in eval and test
print("📉 Anomalies and drift in eval dataset:")
tfdv.display_anomalies(eval_anomalies)

print("📉 Anomalies and drift in test dataset:")
tfdv.display_anomalies(test_anomalies)

📉 Anomalies and drift in eval dataset:


📉 Anomalies and drift in test dataset:


In [26]:
# RAW_DATA_METADATA = tft.DatasetMetadata(schema=schema)
# print("\nConverted to tf.Transform DatasetMetadata object:")

RAW_DATA_FEATURE_SPEC = {
    'Company': tf.io.FixedLenFeature([], tf.string),
    'TypeName': tf.io.FixedLenFeature([], tf.string),
    'Inches': tf.io.FixedLenFeature([], tf.float32),
    'ScreenResolution': tf.io.FixedLenFeature([], tf.string),
    'Cpu': tf.io.FixedLenFeature([], tf.string),
    'Ram': tf.io.FixedLenFeature([], tf.string),
    'Memory': tf.io.FixedLenFeature([], tf.string),
    'Gpu': tf.io.FixedLenFeature([], tf.string),
    'OpSys': tf.io.FixedLenFeature([], tf.string),
    'Weight': tf.io.FixedLenFeature([], tf.string),
    'Price': tf.io.FixedLenFeature([], tf.float32),
}
RAW_DATA_METADATA = tft.tf_metadata.dataset_metadata.DatasetMetadata(
    tft.tf_metadata.schema_utils.schema_from_feature_spec(RAW_DATA_FEATURE_SPEC)
)

In [27]:
def validate_company(company_inputs):
    # --- 1. 'Company' (Categorical) ---
    # Ensure Company is not empty, replace with 'unknown' if it is
    company = tf.strings.lower(tf.strings.strip(company_inputs))
    company = tf.where(
        tf.strings.length(company) > 0,
        company,
        tf.constant("unknown", dtype=tf.string)
    )

    return company

In [28]:
def validate_inches(inches_input):
    inches = tf.cast(inches_input, tf.float32)
    # convert inches feature into buckets (e.g. small <14", medium <15.6", large >=15.6")
    inches_bucket = tf.where(
        inches < 14.0, 0,
        tf.where(inches < 15.6, 1, 2)
    )
    return tf.cast(inches_bucket, tf.float32)

In [29]:
def validate_screen_resolution(screen_resolution_input):
    screen_res_str = tf.strings.strip(screen_resolution_input)
    resolution_parts = tf.strings.split(screen_res_str, "x").to_tensor(default_value="0")
    width = tf.strings.to_number(resolution_parts[:, 0], tf.float32)
    height = tf.strings.to_number(resolution_parts[:, 1], tf.float32)

    return (width, height)

In [30]:
def validate_cpu(cpu_inputs):
    cpu = tf.strings.strip(cpu_inputs)
    cpu = tf.strings.lower(cpu)

    # 1. Extract CPU brand (intel / amd)
    cpu_brand = tf.where(
        tf.strings.regex_full_match(cpu, ".*intel.*"),
        tf.constant("intel"),
        tf.constant("amd")
    )

    # 2. Extract CPU family (e.g. core i5, a9-series, ryzen)
    cpu_family = tf.strings.regex_replace(cpu, r"^(intel|amd)\s+", "")
    cpu_family = tf.strings.regex_replace(cpu_family, r"\s+\d.*", "")  # remove model numbers and GHz

    return cpu_brand, cpu_family


In [31]:
def validate_ram(ram_inputs):
    # Lower and strip
    ram = tf.strings.lower(tf.strings.strip(ram_inputs))

    # Example values: '16gb', '1234tb', '', 'nan'

    # Extract numeric part
    ram_numeric_str = tf.strings.regex_replace(ram, r'[^0-9]', '')

    # Replace empty string with '0'
    is_empty_str = tf.equal(ram_numeric_str, "")
    ram_numeric_str_cleaned = tf.where(is_empty_str, "0", ram_numeric_str)

    # Convert to float
    ram_value = tf.strings.to_number(ram_numeric_str_cleaned, tf.float32)

    # Handle TB → GB (if input had 'tb')
    is_tb = tf.strings.regex_full_match(ram, r'.*tb.*')
    ram_value = tf.where(is_tb, ram_value * 1024.0, ram_value)

    # Handle out-of-range (e.g., 1234TB becomes 1M GB)
    is_invalid_range = tf.logical_or(ram_value <= 0.0, ram_value > 1024.0)
    ram_value = tf.where(is_invalid_range, tf.constant(0.0, tf.float32), ram_value)

    return ram_value


In [32]:
def validate_memory(memory_inputs):
  memory = tf.strings.lower(tf.strings.strip(memory_inputs))

  # Extract numeric part
  num_str = tf.strings.regex_replace(memory, r'[^0-9.]', '')
  size = tf.strings.to_number(num_str, tf.float32)

  # Check if unit is TB → convert to GB
  is_tb = tf.strings.regex_full_match(memory, r'.*tb.*')
  memory_size_gb = tf.where(is_tb, size * 1024.0, size)

  has_ssd = tf.strings.regex_full_match(memory, '.*ssd.*')
  has_hdd = tf.strings.regex_full_match(memory, '.*hdd.*')


  return (memory_size_gb, has_ssd, has_hdd)

In [33]:
def validate_gpu(gpu_input):
    gpu_str = tf.strings.lower(tf.strings.strip(gpu_input))

    is_unknown_value = tf.equal(tf.strings.lower(gpu_str), '###invalid###')
    is_nan = tf.strings.regex_full_match(gpu_str, r'nan')

    is_invalid = tf.logical_or(is_unknown_value, is_nan)
    gpu_str = tf.where(is_invalid, tf.constant("unknown unknown", dtype=tf.string), gpu_str)

    gpu_parts = tf.strings.split(gpu_str, " ").to_tensor(default_value="unknown unknown")
    brand = gpu_parts[:, 0]
    model = gpu_parts[:, 1]

    return brand, model


In [34]:
def validate_price(price_inputs):
    # Step 1: Clean up strings
    price_str = tf.strings.strip(tf.strings.as_string(price_inputs))

    # Step 2: Check for "unknown"
    is_unknown = tf.equal(tf.strings.lower(price_str), 'unknown')

    # Step 3: Convert to number (invalid strings become NaN)
    price_float = tf.strings.to_number(price_str, tf.float32)

    # Step 4: Check for numeric NaN or negatives
    is_nan = tf.math.is_nan(price_float)
    is_negative = price_float < 0.0

    # Step 5: Invalid if "unknown", NaN, or negative
    is_invalid = tf.logical_or(tf.logical_or(is_unknown, is_nan), is_negative)

    # Step 6: Replace invalids with 0.0 (or leave as-is and impute later)
    cleaned_price = tf.where(is_invalid, tf.constant(0.0, tf.float32), price_float)

    return cleaned_price


In [35]:
def preprocessing_fn(inputs):
    outputs = {}

    # --- 1. Company (categorical) ---
    company = validate_company(inputs['Company'])
    outputs['company_xf'] = tft.compute_and_apply_vocabulary(company)

    # --- 2. TypeName (categorical) ---
    type_name = tf.strings.lower(inputs["TypeName"])
    type_name = tf.strings.strip(type_name)
    outputs['typename_xf'] = tft.compute_and_apply_vocabulary(type_name)

    #--- 3. Inches (bucketing) ---
    inches =validate_inches(inputs["Inches"])
    outputs['inches_bucket'] = tf.cast(inches, tf.float32)

    # --- 4. ScreenResolution (numerical) --
    width, height = validate_screen_resolution(inputs["ScreenResolution"])
    outputs["scaled_width"] = tft.scale_to_z_score(width)
    outputs["scaled_height"] = tft.scale_to_z_score(height)

    #--- 5. CPU (categorical) --
    # Process CPU
    cpu_brand, cpu_model = validate_cpu(inputs["Cpu"])
    outputs['cpu_brand_xf'] = tft.compute_and_apply_vocabulary(cpu_brand)
    outputs['cpu_family_xf'] = tft.compute_and_apply_vocabulary(cpu_model)

    #--- 6. RAM (numerical) --
    ram = validate_ram(inputs["Ram"])
    outputs["scaled_ram"] = tft.scale_to_z_score(ram)

    #---7. Memory () ---
    sizes, ssds, hdds, = validate_memory(inputs["Memory"])
    outputs["scaled_memory_size"] = tft.scale_to_z_score(sizes)
    outputs["has_ssd"] = tf.cast(ssds, tf.float32)
    outputs["has_hdd"] = tf.cast(hdds, tf.float32)

    # --- 8. GPU (categorical) ---
    gpu_brand, gpu_model = validate_gpu(inputs['Gpu'])
    outputs['gpu_brand_xf'] = tft.compute_and_apply_vocabulary(gpu_brand)
    outputs['gpu_model_xf'] = tft.compute_and_apply_vocabulary(gpu_model)

    # --- 9. OpSys (categorical) ---
    def normalize_os_single(os_str):
        os_str = tf.strings.lower(tf.strings.strip(os_str))
        is_windows = tf.strings.regex_full_match(os_str, '.*windows.*')
        is_mac = tf.strings.regex_full_match(os_str, '.*mac.*')

        return tf.case([
            (is_windows, lambda: tf.constant("windows")),
            (is_mac, lambda: tf.constant("macos")),
        ], default=lambda: os_str)

    normalized_opsys = tf.map_fn(
        normalize_os_single,
        inputs["OpSys"],
        fn_output_signature=tf.TensorSpec([], tf.string)
    )
    outputs['opsys_xf'] = tft.compute_and_apply_vocabulary(normalized_opsys)

    #---10. Weight (numerical) ---
    weight = tf.strings.lower(inputs['Weight'])
    weight = tf.strings.regex_replace(weight, 'kg', '')
    weight = tf.strings.to_number(weight, tf.float32)
    outputs['scaled_weight'] = tft.scale_to_z_score(weight)

    #---11. Price (numerical) ---
    cleaned_price = validate_price(inputs["Price"])

    # Optionally impute zero values with mean/median using tft
    is_zero = tf.equal(cleaned_price, 0.0)
    non_zero_mean = tft.mean(cleaned_price)
    final_price = tf.where(is_zero, non_zero_mean, cleaned_price)
    outputs["scaled_price"] = tft.scale_to_z_score(final_price)


    return outputs



In [36]:
import apache_beam as beam
import tensorflow_transform.beam as tft_beam
import tempfile
import csv
import numpy as np
import random

# def split_data(element, train_ratio=0.7, eval_ratio=0.15):
#     rnd = random.random()
#     if rnd < train_ratio:
#         tag = 'train'
#     elif rnd < train_ratio + eval_ratio:
#         tag = 'eval'
#     else:
#         tag = 'test'
#     return tag, element

class Split(beam.DoFn):
    def process(self, element):
        try:
            parts = element.strip().split(",")
            if len(parts) != 11:
                return  # skip malformed rows

            Company, TypeName, Inches, ScreenResolution, Cpu, Ram, Memory, Gpu, OpSys, Weight, Price = parts

            def safe_float(val):
                try:
                    return float(val.strip())
                except:
                    return np.nan  # use NaN for invalid values like "unknown"

            yield {
                "Company": Company.strip(),
                "TypeName": TypeName.strip(),
                "Inches": safe_float(Inches),
                "ScreenResolution": ScreenResolution.strip(),
                "Cpu": Cpu.strip(),
                "Ram": Ram.strip(),
                "Memory": Memory.strip(),
                "Gpu": Gpu.strip(),
                "OpSys": OpSys.strip(),
                "Weight": Weight.strip(),
                "Price": safe_float(Price),  # <- safely handle unknown/invalid
            }

        except Exception as e:
            return  # optionally log or skip

In [38]:
def run_pipeline(input_csv, output_prefix, transform_fn_dir=None, analyze=False):
    with beam.Pipeline() as pipeline:
        with tft_beam.Context(temp_dir="./tmp"):
            raw_data = (
                pipeline
                | f"Read CSV {input_csv}" >> beam.io.ReadFromText(input_csv, skip_header_lines=1)
                | f"Parse CSV {input_csv}" >> beam.ParDo(Split())
            )

            dataset = (raw_data, RAW_DATA_METADATA)

            if analyze:
                # Analyze & Transform for training data
                transformed_dataset, transform_fn = (
                    dataset | "Analyze and Transform Train Data" >> tft_beam.AnalyzeAndTransformDataset(preprocessing_fn)
                )
                transformed_data, transformed_metadata = transformed_dataset

                # Save transform_fn
                _ = (
                    transform_fn
                    | "Write TransformFn" >> tft_beam.WriteTransformFn(os.path.join(output_prefix, 'transform_fn'))
                )

            else:
                transform_fn = pipeline | "Read TransformFn" >> tft_beam.ReadTransformFn(transform_fn_dir)

                transformed_data, transformed_metadata = (
                    ((raw_data, RAW_DATA_METADATA), transform_fn)
                    | f"Transform {input_csv}" >> tft_beam.TransformDataset()
                )

            _ = (
                transformed_data
                | "Reshuffle Before Write" >> beam.transforms.util.Reshuffle()
                | f"Write TFRecords {output_prefix}" >> beam.io.WriteToTFRecord(
                    file_path_prefix=output_prefix,
                    file_name_suffix='.gz',
                    coder=tft.coders.example_proto_coder.ExampleProtoCoder(transformed_metadata.schema)
                )
            )


In [39]:
# run_pipeline(
#     input_csv="train_*.csv",
#     output_prefix="train_data",
#     transform_fn_dir=None,
#     analyze=True  # 👈 this is the key
# )

# --- Run for Sample (Analyze + Transform) ---
run_pipeline(
    input_csv="sample.csv",
    output_prefix="train",
    analyze=True
)

# # --- Run for Eval (Transform Only) ---
run_pipeline(
    input_csv="eval.csv",
    output_prefix="eval",
    transform_fn_dir="train/transform_fn",
    analyze=False
)

# # --- Run for Test ---
run_pipeline(
    input_csv="test.csv",
    output_prefix="test",
    transform_fn_dir="train/transform_fn",
    analyze=False
)

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.
Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a warning and probably safe to ignore.
'Counter' object has no attribute 'name'
Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a warning and probably safe to ignore.
'tuple' object has no attribute 'name'
Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a warning and probably safe to ignore.
'Counter' object has no attribute 'name'
Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a warning and probably safe to ignore.
'Counter' object has no attribute 'name'
Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a

value: "\n\013\n\tConst_2:0\022-vocab_compute_and_apply_vocabulary_vocabulary"

value: "\n\013\n\tConst_5:0\022/vocab_compute_and_apply_vocabulary_1_vocabulary"

value: "\n\014\n\nConst_12:0\022/vocab_compute_and_apply_vocabulary_2_vocabulary"

value: "\n\014\n\nConst_15:0\022/vocab_compute_and_apply_vocabulary_3_vocabulary"

value: "\n\014\n\nConst_22:0\022/vocab_compute_and_apply_vocabulary_4_vocabulary"

value: "\n\014\n\nConst_25:0\022/vocab_compute_and_apply_vocabulary_5_vocabulary"

value: "\n\014\n\nConst_28:0\022/vocab_compute_and_apply_vocabulary_6_vocabulary"

value: "\n\013\n\tConst_2:0\022-vocab_compute_and_apply_vocabulary_vocabulary"

value: "\n\013\n\tConst_5:0\022/vocab_compute_and_apply_vocabulary_1_vocabulary"

value: "\n\014\n\nConst_12:0\022/vocab_compute_and_apply_vocabulary_2_vocabulary"

value: "\n\014\n\nConst_15:0\022/vocab_compute_and_apply_vocabulary_3_vocabulary"

value: "\n\014\n\nConst_22:0\022/vocab_compute_and_apply_vocabulary_4_vocabulary"

value: "\n\0

In [41]:
!zip -r /content/train.zip /content/train

  adding: content/train/ (stored 0%)
  adding: content/train/transform_fn/ (stored 0%)
  adding: content/train/transform_fn/transformed_metadata/ (stored 0%)
  adding: content/train/transform_fn/transformed_metadata/schema.pbtxt (deflated 87%)
  adding: content/train/transform_fn/transformed_metadata/asset_map (deflated 88%)
  adding: content/train/transform_fn/transform_fn/ (stored 0%)
  adding: content/train/transform_fn/transform_fn/assets/ (stored 0%)
  adding: content/train/transform_fn/transform_fn/assets/vocab_compute_and_apply_vocabulary_2_vocabulary (stored 0%)
  adding: content/train/transform_fn/transform_fn/assets/vocab_compute_and_apply_vocabulary_vocabulary (stored 0%)
  adding: content/train/transform_fn/transform_fn/assets/vocab_compute_and_apply_vocabulary_3_vocabulary (deflated 3%)
  adding: content/train/transform_fn/transform_fn/assets/vocab_compute_and_apply_vocabulary_5_vocabulary (deflated 28%)
  adding: content/train/transform_fn/transform_fn/assets/vocab_comput

In [42]:
# Read CSV with error handling
df = pd.read_csv("sample.csv")

# Convert Price column to numeric, coerce invalids to NaN
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# Drop rows where price is NaN (optional)
df_clean = df.dropna(subset=["Price"])

# Now compute mean and std safely
print("Mean price:", df_clean["Price"].mean())
print("Std of price:", df_clean["Price"].std())


Mean price: 1636.6564776316397
Std of price: 799.24460642538
